# 03 — Qwen3-1.7B — Few-shot

Execução do experimento **Qwen3-1.7B** em configuração *few-shot* sobre o **conjunto completo de 1.431 autores**.

O notebook utiliza os módulos reutilizáveis de `src/llm/` e `src/evaluation/`, mantendo a configuração do notebook experimental.

In [ ]:
from pathlib import Path
import sys
import numpy as np

current = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [current, *current.parents] if (p / "src").exists()), current)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.llm.qwen_inference import GenerationConfig, carregar_perfis_estruturados, executar_inferencia
from src.evaluation.semantic_matching import (
    SBERT_MODEL_NAME, carregar_qrels, construir_vocabulario, encodar_com_sbert,
    construir_matriz_similaridade, matching_greedy_1_to_1, salvar_npz,
    salvar_csv_avaliacoes_gerais,
)
from src.evaluation.metrics import carregar_perfis_por_documento, avaliar_autor, salvar_csv_metricas, TOP_K_ALVO_METRICAS

print(f"Raiz do projeto: {PROJECT_ROOT}")

In [ ]:
DATA_DIR = PROJECT_ROOT / "data" / "processed"
RESULTS_DIR = PROJECT_ROOT / "results" / "qwen_few_shot" / "qwen3_1_7b"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

PERFIS_ESTRUTURADOS = DATA_DIR / "perfis_estruturados_qwen.json"
PERFIS_DOCUMENTO = DATA_DIR / "perfis_documento_qwen.json"
QRELS = DATA_DIR / "ground_truth" / "LExR-prof-qrels_filtrado"

TAGS_BRUTAS = RESULTS_DIR / "tags_brutas_qwen3_1_7b.json"
RANKINGS = RESULTS_DIR / "ranking_qwen3_1_7b.json"
CHECKPOINT = RESULTS_DIR / "tags_brutas_qwen3_1_7b_checkpoint.json"
CSV_METRICAS = RESULTS_DIR / "metricas_qwen3_1_7b_por_autor.csv"
CSV_AVALIACOES = RESULTS_DIR / "avaliacoes_gerais_autores_qwen3_1_7b.csv"
SIM_DIR = RESULTS_DIR / "sim_matrices"
SIM_DIR.mkdir(parents=True, exist_ok=True)

for nome, caminho in {
    "Perfis estruturados": PERFIS_ESTRUTURADOS,
    "Perfis por documento": PERFIS_DOCUMENTO,
    "Qrels": QRELS,
}.items():
    print(f"{nome:24s}: {'OK' if caminho.exists() else 'não encontrado'}")

## Configuração do modelo

Valores preservados do experimento original Qwen3-1.7B no conjunto completo. Neste modelo, `enable_thinking=False` e `min_p=0.0` são definidos explicitamente.

In [ ]:
config = GenerationConfig(
    model_name="Qwen/Qwen3-1.7B",
    mode="few-shot",
    n_tags=30,
    max_publicacoes=50,
    batch_size=4,
    do_sample=True,
    temperature=0.7,
    top_p=0.8,
    top_k=20,
    min_p=0.0,
    enable_thinking=False,
    repetition_penalty=1.0,
    max_new_tokens=1024,
    max_input_tokens=16384,
    seed=42,
    checkpoint_every_authors=12,
)
config

## Carregamento dos 1.431 autores

In [ ]:
perfis = carregar_perfis_estruturados(PERFIS_ESTRUTURADOS)
print(f"Autores carregados: {len(perfis):,}")
if len(perfis) != 1431:
    print("[Aviso] O experimento original utiliza 1.431 autores. Verifique o arquivo de entrada.")

## Inferência few-shot

In [ ]:
tags_brutas, rankings = executar_inferencia(
    perfis,
    config,
    caminho_saida=TAGS_BRUTAS,
    caminho_checkpoint=CHECKPOINT,
    caminho_ranking=RANKINGS,
)
print(f"Autores processados: {len(tags_brutas):,}")

## Avaliação semântica

SBERT `paraphrase-multilingual-mpnet-base-v2`, embeddings L2-normalizados, similaridade de cosseno, *global greedy* 1-para-1 e limiar 0,75.

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

THRESHOLD_SBERT = 0.75
BATCH_SBERT = 256
SBERT_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

gt_norm, gt_original = carregar_qrels(QRELS)
perfis_doc = carregar_perfis_por_documento(PERFIS_DOCUMENTO)

vocab = construir_vocabulario(
    rankings=rankings,
    gt_norm=gt_norm,
    perfis_doc_semantico=perfis_doc,
    top_k_pred=TOP_K_ALVO_METRICAS,
)
print(f"Vocabulário SBERT: {len(vocab):,} strings")

modelo_sbert = SentenceTransformer(SBERT_MODEL_NAME, device=SBERT_DEVICE)
cache_emb = encodar_com_sbert(modelo_sbert, vocab, batch_size=BATCH_SBERT, device=SBERT_DEVICE)
if SBERT_DEVICE == "cuda":
    del modelo_sbert
    torch.cuda.empty_cache()

In [ ]:
dados_por_autor = {}
matching_por_autor = {}

for autor, ranking in rankings.items():
    if autor not in gt_norm:
        continue
    dados = construir_matriz_similaridade(
        autor=autor,
        ranking=ranking,
        gt_norm_autor=gt_norm[autor],
        cache_emb=cache_emb,
        top_k=TOP_K_ALVO_METRICAS,
    )
    if dados is None:
        continue

    matched_weights, matched_idx, matched_sims = matching_greedy_1_to_1(
        dados["sim"], dados["gold_weights"], theta=THRESHOLD_SBERT
    )
    salvar_npz(dados, matched_idx, matched_weights, matched_sims, SIM_DIR)

    dados_por_autor[autor] = dados
    matching_por_autor[autor] = {
        "matched_weights": matched_weights,
        "matched_idx": matched_idx,
        "matched_sims": matched_sims,
    }

print(f"Autores avaliados contra o gabarito: {len(dados_por_autor):,}")

## Métricas

In [ ]:
metricas_por_autor = {}

for autor, dados in dados_por_autor.items():
    metricas_por_autor[autor] = avaliar_autor(
        dados_autor=dados,
        matched_weights=matching_por_autor[autor]["matched_weights"],
        docs_ngrams_autor=perfis_doc.get(autor, {}),
        docs_campos_autor=perfis_doc.get(autor, {}),
        cache_emb=cache_emb,
        theta_cov=0.75,
    )

salvar_csv_metricas(metricas_por_autor, CSV_METRICAS, modelo="Qwen3-1.7B")
salvar_csv_avaliacoes_gerais(
    dados_por_autor=dados_por_autor,
    matching_por_autor=matching_por_autor,
    gt_original=gt_original,
    caminho=CSV_AVALIACOES,
    modelo="Qwen3-1.7B",
    rank_max=20,
)

print(f"Métricas: {CSV_METRICAS}")
print(f"Avaliação detalhada: {CSV_AVALIACOES}")

## Resumo agregado

In [ ]:
if metricas_por_autor:
    nomes = list(next(iter(metricas_por_autor.values())).keys())
    medias = {nome: float(np.mean([m[nome] for m in metricas_por_autor.values()])) for nome in nomes}
    for nome, valor in medias.items():
        print(f"{nome:20s}: {valor:.4f}")
else:
    print("Nenhuma métrica disponível.")

## Saídas

O notebook gera tags brutas, ranking posicional, matrizes `.npz`, métricas por autor e o CSV detalhado de pareamento. 